# Video Classification with a 3D CNN (UCF101)

A demo of video classification using a `Conv3D` neural network based on [A Closer Look at Spatiotemporal Convolutions for Action Recognition](https://ieeexplore.ieee.org/document/8578773) by D. Tran et al. (2017).

### What students should learn
1. A video is a tensor with shape **(time, height, width, channels)**.
2. `Conv2D` learns spatial patterns; `Conv3D` can learn **spatiotemporal** patterns.
3. A `3×3×3` kernel spans **3 frames × 3 pixels × 3 pixels**.
4. We can train a small video classifier end-to-end from raw video clips.

We use a public **5-class subset of UCF101** rather than downloading the complete UCF101 dataset.

> **Colab:** For faster training, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Check TensorFlow and GPU
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU(s):", tf.config.list_physical_devices("GPU"))

## 1. Download a small UCF101 subset

This is the same 5-class UCF101 subset used by Keras video-classification examples.
It contains `train/`, `test/`, `train.csv`, and `test.csv`.

This avoids the old `storage.googleapis.com/thumos14_files/UCF101_videos.zip`
URL that may return HTTP 403.

In [ ]:
!wget -q --show-progress \
  https://github.com/sayakpaul/Action-Recognition-in-TensorFlow/releases/download/v1.0.0/ucf101_top5.tar.gz

!tar -xf ucf101_top5.tar.gz

!ls -lh

## 2. Inspect the dataset

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Training videos:", len(train_df))
print("Test videos:", len(test_df))
print("\nClasses:", sorted(train_df["tag"].unique()))
display(train_df.head())

print("\nTraining examples per class:")
display(train_df["tag"].value_counts().sort_index())

## 3. Video → fixed-size clip

Neural networks need a fixed tensor shape. We represent every video by:

**16 frames × 96 × 96 pixels × 3 color channels**

Rather than taking only the first 16 frames, we **uniformly sample 16 frames across the video**.

In [ ]:
NUM_FRAMES = 16
IMG_SIZE = 96

def load_video(path, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
    # Uniformly sample frames and return a (T,H,W,C) float32 tensor.
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return np.zeros((num_frames, img_size, img_size, 3), dtype=np.float32)

    frame_ids = np.linspace(0, max(frame_count - 1, 0), num_frames).astype(int)

    frames = []
    for frame_id in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
        ok, frame = cap.read()

        if not ok:
            if frames:
                frame = (frames[-1] * 255).astype(np.uint8)
                frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            else:
                frame = np.zeros((img_size, img_size, 3), dtype=np.uint8)

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (img_size, img_size))
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    cap.release()
    return np.stack(frames)

example_row = train_df.sample(1, random_state=SEED).iloc[0]
example_path = os.path.join("train", example_row["video_name"])
example_clip = load_video(example_path)

print("Class:", example_row["tag"])
print("Clip shape:", example_clip.shape)

### Visualize sampled frames

The temporal dimension is explicit: the CNN receives a sequence of frames, not a single image.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), np.linspace(0, NUM_FRAMES-1, 8).astype(int)):
    ax.imshow(example_clip[i])
    ax.set_title(f"Frame {i}")
    ax.axis("off")

plt.suptitle(f"Uniformly sampled frames — {example_row['tag']}")
plt.tight_layout()
plt.show()

## 4. Labels and train/validation split

The archive already provides train and test sets. We split part of the training set
into a validation set.

`FAST_DEMO=True` limits the amount of data so the notebook is convenient for a live lecture.
Set it to `False` to use the whole provided training subset.

In [ ]:
classes = sorted(train_df["tag"].unique())
class_to_id = {name: i for i, name in enumerate(classes)}
id_to_class = {i: name for name, i in class_to_id.items()}

print(class_to_id)

train_part, val_part = train_test_split(
    train_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_df["tag"]
)

FAST_DEMO = True
MAX_TRAIN_PER_CLASS = 40
MAX_VAL_PER_CLASS = 10
MAX_TEST_PER_CLASS = 15

def limit_per_class(df, n):
    if n is None:
        return df.reset_index(drop=True)

    parts = []
    for _, group in df.groupby("tag"):
        parts.append(group.sample(min(len(group), n), random_state=SEED))
    return pd.concat(parts).reset_index(drop=True)

if FAST_DEMO:
    train_part = limit_per_class(train_part, MAX_TRAIN_PER_CLASS)
    val_part = limit_per_class(val_part, MAX_VAL_PER_CLASS)
    test_part = limit_per_class(test_df, MAX_TEST_PER_CLASS)
else:
    test_part = test_df.reset_index(drop=True)

print("Train:", len(train_part))
print("Validation:", len(val_part))
print("Test:", len(test_part))

## 5. Build an on-the-fly video data loader

Loading every decoded video into RAM at once is wasteful. This `Sequence` reads and decodes
only the videos needed for the current mini-batch.

In [ ]:
class VideoSequence(tf.keras.utils.Sequence):
    def __init__(self, dataframe, root_dir, batch_size=4, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.df = dataframe.reset_index(drop=True).copy()
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, batch_index):
        inds = self.indices[
            batch_index * self.batch_size:
            (batch_index + 1) * self.batch_size
        ]

        batch_x = []
        batch_y = []

        for i in inds:
            row = self.df.iloc[i]
            path = os.path.join(self.root_dir, row["video_name"])
            batch_x.append(load_video(path))
            batch_y.append(class_to_id[row["tag"]])

        return (
            np.asarray(batch_x, dtype=np.float32),
            np.asarray(batch_y, dtype=np.int32)
        )

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

BATCH_SIZE = 4

train_seq = VideoSequence(train_part, "train", BATCH_SIZE, shuffle=True)
val_seq   = VideoSequence(val_part,   "train", BATCH_SIZE, shuffle=False)
test_seq  = VideoSequence(test_part,  "test",  BATCH_SIZE, shuffle=False)

x_batch, y_batch = train_seq[0]
print("Input batch:", x_batch.shape)
print("Labels:", y_batch.shape)

## 6. Build a true 3D CNN

Compare:

```python
Conv2D(16, (3, 3))       # height × width
Conv3D(16, (3, 3, 3))    # time × height × width
```

Here the input shape is:

`(16, 96, 96, 3) = (time, height, width, channels)`

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

NUM_CLASSES = len(classes)

model = keras.Sequential([
    layers.Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv3D(
        16, kernel_size=(3, 3, 3),
        strides=(1, 2, 2),
        padding="same"
    ),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPool3D(pool_size=(2, 2, 2)),

    layers.Conv3D(32, (3, 3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPool3D(pool_size=(2, 2, 2)),

    layers.Conv3D(64, (3, 3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPool3D(pool_size=(2, 2, 2)),

    layers.GlobalAveragePooling3D(),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Key teaching point

After the first `Conv3D`, each learned filter can respond to a pattern that changes across
nearby frames. This allows the network to model **appearance + short-term motion**.

The model is intentionally small. The goal is to make the mechanism visible, not to reach
state-of-the-art UCF101 accuracy.

## 7. Train

For a live demonstration, **3–5 epochs** are enough to show the training process.
For a better result, increase `EPOCHS` and set `FAST_DEMO=False`.

A GPU runtime is strongly recommended.

In [ ]:
EPOCHS = 5

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 8. Plot the learning curves

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(7, 4))
plt.plot(history_df["accuracy"], marker="o", label="train")
plt.plot(history_df["val_accuracy"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["loss"], marker="o", label="train")
plt.plot(history_df["val_loss"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 9. Evaluate on unseen test videos

In [ ]:
test_loss, test_accuracy = model.evaluate(test_seq, verbose=1)
print(f"Test accuracy: {test_accuracy:.3f}")

## 10. Predict one video

We display the original video and then run the exact same 16-frame preprocessing pipeline.

In [ ]:
from IPython.display import Video, display

sample = test_part.sample(1, random_state=7).iloc[0]
sample_path = os.path.join("test", sample["video_name"])

print("Ground truth:", sample["tag"])
display(Video(sample_path, embed=True, width=500))

clip = load_video(sample_path)
probs = model.predict(clip[None, ...], verbose=0)[0]

order = np.argsort(probs)[::-1]

print("\nPredictions:")
for i in order:
    print(f"{id_to_class[i]:20s} {probs[i]:.3f}")

plt.figure(figsize=(8, 4))
plt.bar([id_to_class[i] for i in order], probs[order])
plt.ylabel("Probability")
plt.xticks(rotation=30, ha="right")
plt.show()

## 11. Inspect the tensor shapes

A single-image classifier receives:

`(batch, height, width, channels)`

Our video classifier receives:

`(batch, time, height, width, channels)`

For this notebook:

`(batch, 16, 96, 96, 3)`

A `3×3×3` convolutional kernel moves through **time, height, and width**.

In [ ]:
x = tf.zeros((1, NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3))

print("Input".ljust(28), x.shape)

for layer in model.layers:
    x = layer(x, training=False)
    print(layer.name.ljust(28), x.shape)

## References

- UCF101: Soomro, Zamir, and Shah, *UCF101: A Dataset of 101 Human Actions Classes From Videos in The Wild*.
- Keras video-classification examples use this public 5-class UCF101 subset.
- A Closer Look at Spatiotemporal Convolutions for Action Recognition by D. Tran et al. (2017)